# 🤝 Notebook: Agents — *Part 4 of 4*

*This is the fourth of four notebooks in this chapter: 1) MCP → 2) Agents → 3) Context Compression & Memory → **4) Subagents**. Start with `06_1_mcp.ipynb` if you haven't yet.*

In this notebook we build a **multi-agent system**: one main agent that delegates specialized subtasks to other agents, instead of doing everything itself.

## 📚 Sources

- [LangChain Documentation: Multi-agent](https://docs.langchain.com/oss/python/langchain/multi-agent)

## Why use more than one agent?

A single `create_agent` with a big pile of tools and a long system prompt can go a long way - and often that's the right call; not every task needs multiple agents. But it starts to break down as things grow:

- **Too many tools confuse the model.** With ten unrelated tools in one prompt, the model more often picks the wrong one.
- **Specialized context doesn't fit cleanly.** A tool that needs 2000 tokens of domain-specific instructions crowds out everything else in one shared system prompt.
- **Intermediate work clutters the conversation.** A research subtask might take five tool calls to answer one question - your main agent's context (and your tests, and your students reading the trace) shouldn't have to wade through all five just to get the answer.

The [LangChain docs](https://docs.langchain.com/oss/python/langchain/multi-agent) describe several multi-agent patterns (Subagents, Handoffs, Skills, Router, Custom workflow). This notebook covers **Subagents** - the simplest and most broadly useful one: a main agent treats other agents as tools it can call. All routing stays under the main agent's control, and each subagent gets a completely fresh, isolated context.

In [1]:
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv()

LLM_HOST = os.environ["LLM_HOST"]
LLM_URL = f"http://{LLM_HOST}:11434"
LLM_REASONING = "gemma4:26b"

llm = ChatOllama(model=LLM_REASONING, base_url=LLM_URL, temperature=0)

## The pattern: an agent, wrapped as a tool

There's no special "subagent" class - a subagent is just another `create_agent`, and we hand it to the main agent by wrapping it in a plain `@tool`-decorated function that calls `.invoke()` on it and returns the final message. From the main agent's point of view, `delegate_to_weather_researcher` is a tool exactly like `get_weather` was in `06_2_agents.ipynb` - it just happens to be backed by an entire other agent instead of one function call.

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool


def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    weather = {"Tokyo": "sunny, 18°C", "London": "rainy, 12°C", "New York": "cloudy, 15°C"}
    return weather.get(city, "unknown city")


# The subagent: its own model, its own tools, its own system prompt.
# It does NOT inherit anything from the main agent - if it needs instructions, they go here.
weather_subagent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a weather researcher. Use get_weather for each city you're asked about, then report back concisely as plain text.",
)


@tool
def delegate_to_weather_researcher(question: str) -> str:
    """Delegate a weather-related question to a specialized weather research subagent."""
    result = weather_subagent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content  # only the final answer goes back - not the subagent's own tool calls


main_agent = create_agent(
    model=llm,
    tools=[delegate_to_weather_researcher],
    system_prompt="You are a travel planning assistant. Delegate weather questions to the weather researcher.",
)

In [3]:
result = main_agent.invoke({"messages": [{"role": "user", "content": "Should I pack an umbrella for London or Tokyo?"}]})

for m in result["messages"]:
    tool_calls = getattr(m, "tool_calls", None)
    if tool_calls:
        print(f"{type(m).__name__}: calls {[tc['name'] + str(tc['args']) for tc in tool_calls]}")
    else:
        content = m.content if isinstance(m.content, str) else str(m.content)
        print(f"{type(m).__name__}: {content!r}")

HumanMessage: 'Should I pack an umbrella for London or Tokyo?'
AIMessage: calls ["delegate_to_weather_researcher{'question': 'What is the expected weather and chance of rain in London and Tokyo for my upcoming trip?'}"]
ToolMessage: 'In London, it is currently rainy with a temperature of 12°C. In Tokyo, it is sunny with a temperature of 18°C.'
AIMessage: "You should definitely pack an umbrella for **London**, as it is currently rainy there. For **Tokyo**, you likely won't need one for rain since it is currently sunny, though you might want sunglasses!"


Look at the message trace: there's exactly **one** `AIMessage` calling `delegate_to_weather_researcher`, and exactly **one** `ToolMessage` coming back - even though, under the hood, `weather_subagent` may have called `get_weather` for both London *and* Tokyo to answer the question. Those two internal tool calls happened in the subagent's own, separate message list - the main agent's context never saw them. This is the multi-agent version of what `07_2_agentic_rag.ipynb` did with retrieval: keeping detailed intermediate work out of the context that matters for the final answer.

One more thing worth noting: subagents are **stateless** here. Every call to `delegate_to_weather_researcher` builds a brand-new `{"messages": [...]}` list from scratch - the subagent has no memory of any previous delegation. That's a deliberate, simple default; if you needed a subagent with its own persistent memory, you'd give it its own checkpointer (`06_3_context_memory.ipynb`) the same way you would any other agent.

## Choosing between several subagents

The real power of this pattern shows up with more than one subagent: the main agent has to decide *which* specialist a given question belongs to - and, since tool calls can be parallel, it can delegate to more than one at the same time when a question needs both.

In [4]:
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another."""
    rates = {("USD", "JPY"): 155.0, ("USD", "GBP"): 0.79}
    rate = rates.get((from_currency, to_currency))
    if rate is None:
        return "rate not available"
    return f"{amount} {from_currency} = {amount * rate:.2f} {to_currency}"


finance_subagent = create_agent(
    model=llm,
    tools=[convert_currency],
    system_prompt="You are a finance assistant. Report back concisely.",
)


@tool
def delegate_to_finance_assistant(question: str) -> str:
    """Delegate a currency-conversion or finance-related question to a specialized finance subagent."""
    result = finance_subagent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content


multi_specialist_agent = create_agent(
    model=llm,
    tools=[delegate_to_weather_researcher, delegate_to_finance_assistant],
    system_prompt="You are a travel planning assistant. Delegate to the right specialist subagent for each part of the question.",
)

result = multi_specialist_agent.invoke({"messages": [{"role": "user", "content":
    "I'm going to Tokyo with 500 USD. What's the weather like, and how much is that in JPY?"}]})

for m in result["messages"]:
    tool_calls = getattr(m, "tool_calls", None)
    if tool_calls:
        print(f"{type(m).__name__}: calls {[tc['name'] + str(tc['args']) for tc in tool_calls]}")
    else:
        content = m.content if isinstance(m.content, str) else str(m.content)
        print(f"{type(m).__name__}: {content!r}")

HumanMessage: "I'm going to Tokyo with 500 USD. What's the weather like, and how much is that in JPY?"
AIMessage: calls ["delegate_to_weather_researcher{'question': 'What is the current weather like in Tokyo?'}", "delegate_to_finance_assistant{'question': 'How much is 500 USD in JPY?'}"]
ToolMessage: 'The current weather in Tokyo is sunny and 18°C.'
ToolMessage: '500 USD is 77,500.00 JPY.'
AIMessage: 'The current weather in Tokyo is sunny and 18°C. Also, 500 USD is approximately 77,500.00 JPY. Enjoy your trip!'


The main agent split the question in two and delegated each half to the right specialist - in a single `AIMessage` with two parallel tool calls, not two separate round trips. Neither subagent ever saw the other's territory: `weather_subagent` only knows about `get_weather`, `finance_subagent` only knows about `convert_currency`.

## When *not* to reach for this

Multi-agent delegation isn't free: each delegation is at least one extra model call (the subagent's own reasoning), so it adds latency and cost. It's a good trade when the subtask genuinely needs isolation - specialized instructions, many intermediate tool calls, or context you don't want polluting the main conversation. For a single, simple tool call, delegating to a whole subagent just to call one function is pure overhead - just give the main agent the tool directly, like every notebook before this one did.

`chapter/09_deep_agents/` picks this exact pattern back up: `deepagents`' built-in `task` tool does what `delegate_to_weather_researcher` does here, just pre-built and wired up for you (along with a virtual filesystem and task planning) - worth comparing side by side once you get there.

## Exercise: A three-specialist coordinator

Add a third subagent, `trivia_subagent`, that answers general knowledge questions (no tool needed - just its own model and a system prompt telling it to answer briefly from what it knows). Give the main agent all three subagents (`weather`, `finance`, `trivia`) and ask it something that needs all three at once, e.g. *"I'm visiting Tokyo with 500 USD - what's the weather, how much is that in JPY, and what's one famous landmark I should see?"* Check the trace to confirm all three get delegated to correctly.

In [5]:
# Insert code here...

<details>
<summary><b>Show solution</b></summary>

```python
trivia_subagent = create_agent(
    model=llm,
    system_prompt="You are a trivia assistant. Answer general knowledge questions briefly and factually from what you know.",
)


@tool
def delegate_to_trivia_assistant(question: str) -> str:
    """Delegate a general knowledge/trivia question to a specialized trivia subagent."""
    result = trivia_subagent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content


coordinator = create_agent(
    model=llm,
    tools=[delegate_to_weather_researcher, delegate_to_finance_assistant, delegate_to_trivia_assistant],
    system_prompt="You are a travel planning assistant. Delegate to the right specialist subagent for each part of the question.",
)

result = coordinator.invoke({"messages": [{"role": "user", "content":
    "I'm visiting Tokyo with 500 USD - what's the weather, how much is that in JPY, "
    "and what's one famous landmark I should see?"}]})

for m in result["messages"]:
    tool_calls = getattr(m, "tool_calls", None)
    if tool_calls:
        print(f"{type(m).__name__}: calls {[tc['name'] + str(tc['args']) for tc in tool_calls]}")
    else:
        content = m.content if isinstance(m.content, str) else str(m.content)
        print(f"{type(m).__name__}: {content!r}")
```

</details>